# LLM Parameters Deep Dive
## Context Window, Temperature, Top-P, Top-K & More

# Setup

In [57]:
from ollama import Client

client = Client()

MODEL = "llama3"

def ask(prompt, **kwargs):
    """Helper: send a user message and return the text response."""
    response = client.generate(
        model=MODEL,
        prompt=prompt,
        options=kwargs
    )
    return(response["response"])

---
## 1. Context Window

### What is it?
The **context window** is the maximum number of **tokens** the model can process at once your input + its output combined.

### It consists of: 

**System Message:** special message that sets the **persona, tone, constraints, and behavior** of the model. 

**User Message:** Actual task or query to LLM

**Assistant's Response:** Response from the LLM


| Model | Context Window |
|-------|---------------|
| Anthropic Claude Sonnet 4| 200K tokens |
| OpenAI GPT-5.5 | 1M tokens |
| Anthropic Claude Opus 4.7 | 1M tokens |
| Google Gemini 1.5 Pro | 2M tokens |

### Why it matters:
- **Too short** → model does not have much context to answer accurately
- **Longer** → more expensive, slower, lost-in-the-middle, but can handle full documents

In [18]:
conversation_history = []

def chat(user_message):
    """Multi-turn chat that accumulates context."""
    conversation_history.append({"role": "user", "content": user_message})
    response = client.chat(
        model=MODEL,
        messages=conversation_history
    )
    print(f"Tokens Count: {response.prompt_eval_count + response.eval_count}")
    reply = response["message"]["content"]
    conversation_history.append(response["message"])
    return reply

# Turn 1 — introduce a fact
r1 = chat("My favorite programming language is Python. Remember that.")
print(f"Turn 1 → {r1}\n")

# Turn 2 — ask something unrelated
r2 = chat("What is 2 + 2?")
print(f"Turn 2 → {r2}\n")

# Turn 3 — test context memory
r3 = chat("What is my favorite programming language?")
print(f"Turn 3 → {r3}")
print(f"\n📊 Total messages in context: {len(conversation_history)}")

Tokens Count: 67
Turn 1 → I've got it! Your favorite programming language is indeed Python. I'll keep that in mind as we chat and explore various topics together. Python is a great language, by the way - what do you like most about it?

Tokens Count: 119
Turn 2 → A classic question!

In Python (remember, your favorite programming language!), the answer to 2 + 2 would be... (drumroll please)... 4!

Tokens Count: 146
Turn 3 → I remember! Your favorite programming language is Python.

📊 Total messages in context: 6


---
## 2. Temperature

### What is it?
Temperature controls the **randomness** of the model's output 

| Temperature | Behavior | Best for |
|-------------|----------|----------|
| `0.0` | Fully deterministic, always picks highest-prob token | Code, math, facts |
| `0.3–0.5` | Slightly creative, mostly consistent | Summarization, Q&A |
| `0.7–1.0` | Balanced creativity | General chat, writing |
| `1.0–2.0` | Very random, surprising, sometimes incoherent | Brainstorming, poetry |

### Intuition:
- **Low temp** = model is "confident" and conservative
- **High temp** = model is "adventurous" explores unlikely tokens

In [20]:
# same prompt at different temperatures
prompt = "Write a one-sentence tagline for a coffee shop."

for temp in [0.0, 0.5, 1.0, 1.5]:
    result = ask(prompt, temperature=temp, num_predict=60)
    print(f"🌡️  temp={temp}: {result}")

🌡️  temp=0.0: "Fuel your day with a cup of joy at Brewed Awakening, where every sip is a wake-up call to the best part of your morning!"
🌡️  temp=0.5: "Fuel your day with a cup of joy at [Coffee Shop Name], where every sip is a chance to start anew."
🌡️  temp=1.0: "Wake up to the perfect cup: where every sip feels like a fresh start."
🌡️  temp=1.5: "Warming hearts and hands, one cup at a time."


---
## 3. Top-P

### What is it?
Instead of sampling from **all** possible tokens, **Top-P** restricts sampling to the **smallest set of tokens** whose **cumulative probability ≥ P**.

| Top-P | Effect |
|-------|--------|
| `1.0` | Consider 100% of token mass (no filtering) |
| `0.9` | Consider top tokens summing to 90% probability |
| `0.5` | Very focused — only the most likely tokens |
| `0.1` | Extremely constrained — almost deterministic |


In [24]:
# Live demo: Top-P effect
prompt = "Give me an excuse for being late to work.'"

for top_p in [0.1, 0.5, 0.95]:
    result = ask(prompt, top_p=top_p, temperature=1.0, num_predict=100)
    print(f"\n🎯 top_p={top_p}:")
    print(f"   {result}")


🎯 top_p=0.1:
   Here's a plausible excuse:

"I'm so sorry I'm running behind schedule. I had a bit of a unexpected situation this morning. On my way out the door, I noticed that my car was making a strange noise and wouldn't start. I tried jump-starting it, but nothing seemed to work. Luckily, a neighbor happened to be walking by and offered to give me a ride to work. We got stuck in traffic on the way here, which didn't help matters. But I'm

🎯 top_p=0.5:
   Here's a plausible excuse:

"I'm so sorry I'm running behind schedule. I had a bit of a situation on my hands this morning. As I was getting ready, I noticed that my car wouldn't start. I tried jump-starting it, but nothing seemed to work. Luckily, a neighbor happened to be walking by and offered to give me a ride to the office. Unfortunately, traffic was worse than expected, and we got stuck in a jam on the highway. By the time we

🎯 top_p=0.95:
   Here's a plausible-sounding excuse:

"I'm so sorry I'm running behind schedule. I

---
## 4. Top-K

### What is it?
**Top-K** sampling limits the model to only the **K most likely tokens** at each step regardless of their actual probability values.

| Top-K | Effect |
|-------|--------|
| `1` | Always pick the single most likely token (greedy decoding) |
| `10` | Choose from top 10 candidates |
| `50` | Moderate vocabulary — common default |
| `∞ / disabled` | All tokens eligible |

### Top-K vs Top-P:
| Top-K | Top-P |
|---|---|
| Fixed number of tokens | Dynamic number of tokens |
| Simpler |  Smarter/Adaptive |
| Less commonly preferred today | Most modern models use this |

> Top-P is generally preferred because it **adapts**: when the model is confident, fewer tokens are in the nucleus. Top-K is always a fixed count.

In [25]:
# Live demo: Top-K effect
prompt = "Give me an excuse for being late to work.'"

for top_k in [1, 10, 50]:
    result = ask(prompt, top_k=top_k, temperature=1.0, num_predict=80)
    print(f"\n🎯 top_k={top_k}:")
    print(f"   {result}")


🎯 top_k=1:
   Here's a plausible excuse:

"I'm so sorry I'm running behind schedule. I had a bit of a unexpected situation this morning. On my way out the door, I noticed that my car was making a strange noise and wouldn't start. I tried jump-starting it, but nothing seemed to work. Luckily, a neighbor happened to be walking by and offered to give me a ride to work

🎯 top_k=10:
   Here's a polite and reasonable-sounding excuse for being late to work:

"I'm so sorry I'm running a bit behind schedule this morning. I had a unexpected issue with my commute that slowed me down - there was an accident on the highway and traffic was backed up all the way to the exit. By the time I arrived, it took me an extra 20 minutes just to get into the

🎯 top_k=50:
   Here's a plausible excuse:

"Hi [Manager's Name], I'm so sorry I'm running a bit behind schedule. I had a bit of a mishap on my way to work this morning. I was driving to the office and suddenly realized I'd left my favorite coffee mug at 

---
## 5. Max Tokens

### What is it?
`max_tokens` sets the **maximum number of tokens the model can generate** in its response. The model stops either when it naturally finishes or hits this limit.

| Use Case | Recommended max_tokens |
|----------|------------------------|
| Yes/No answers | 5–20 |
| Short answers / classification | 50–200 |
| Summaries / Q&A | 200–500 |
| Long-form writing | 1000–4096 |
| Full document generation | 8192+ |

In [35]:
# ── See the effect of different max_tokens 
prompt = "Write a 200-word essay on the ocean."

for max_t in [20, 100, 300]:
    response = client.chat(
        model=MODEL,
        messages=[
            {
            "role": "user",
            "content": prompt
        }],
        options={"num_predict":max_t}
    )
    text = response["message"]["content"]
    stop = response.done_reason
    print(f"\n🛑 max_tokens={max_t} | stop_reason={stop}")
    print(text)


🛑 max_tokens=20 | stop_reason=length
The ocean, often referred to as the "blue planet's" lifeblood, is a vast and

🛑 max_tokens=100 | stop_reason=length
The ocean is a vast and wondrous place, covering over 70% of our planet's surface. It is home to an incredible array of marine life, from tiny plankton to massive blue whales. The ocean plays a crucial role in regulating the Earth's climate, providing half of the oxygen we breathe, and serving as a vital source of food for millions of people.

The ocean's depths are a mystery waiting to be unraveled. From the sunlit surface waters to the darkest depths, each

🛑 max_tokens=300 | stop_reason=stop
The ocean is a vast and mysterious expanse that covers over 70% of our planet's surface. It is a vital component of our ecosystem, providing half of the oxygen we breathe and serving as a source of food for billions of people around the world. The ocean's depths are home to an incredible array of marine life, from tiny plankton to massive blue 

---
## 6. Stop Sequences

### What is it?
**Stop sequences** are strings that tell the model to **stop generating** as soon as it produces them. They are **not included** in the output.

Common use cases:
- Parsing structured outputs (stop at `"###"` or `"</answer>"`)
- Multi-turn role-play (stop at `"Human:"`)
- Getting just one item from a list (stop at `"\n"`)

In [37]:
# ── Use stop sequences to extract just the answer ────────────────────────────
prompt = """Answer the question, then explain your reasoning.

Question: What is 17 * 6?
Answer: """

# Without stop sequence — get full response
full = ask(prompt, num_predict=200)
print("Without stop sequence:")
print(full)
print()

# With stop sequence — only get the answer number
concise = ask(prompt, stop=["\n"])

print("With stop_sequences=['\\n']:")
print(concise)

Without stop sequence:
Answer: 102

Reasoning: To calculate 17 * 6, I multiplied the two numbers together. The standard procedure for multiplication is to multiply the numbers separately and then combine them. In this case:

* 7 x 6 = 42
* 10 (the tens place of 17) x 6 = 60
* Add the results: 42 + 60 = 102

So, 17 * 6 equals 102.

With stop_sequences=['\n']:
The answer to 17 * 6 is 102.


---
## 7. Combining Parameters - Real World Usecases

Different tasks call for different parameter combinations. Here are **production-ready recipes**:

In [55]:
def get_config(task_type: str) -> dict:
    """
    Returns recommended parameter configs for common task types.
    task_type: 'code' | 'creative' | 'qa' | 'extraction' | 'chat'
    """
    configs = {
        "code":       {"temperature": 0.0, "top_p": 1.0, "num_predict": 1024},
        "creative":   {"temperature": 1.2, "top_p": 0.95, "top_k": 100, "num_predict": 800},
        "qa":         {"temperature": 0.2, "top_p": 0.8, "num_predict": 300, stop: ["\n\n"]},
        "extraction": {"temperature": 0.0, "top_p": 1.0, "num_predict": 200},
        "chat":       {"temperature": 0.7, "top_p": 0.9, "num_predict": 512},
    }
    return configs.get(task_type, configs["chat"])

In [40]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  USECASE 1: Precise Code Generation                          ║
# ║  Goal: Deterministic, correct, no hallucination             ║
# ╚══════════════════════════════════════════════════════════════╝

code_response = client.chat(
    model=MODEL,
    messages=[
        {
        "role": "system",
        "content": "You are an expert Python developer. Return only the Python code with no explanation."
    },{
        "role": "user",
        "content": "Write a Python function that checks if a string is a palindrome."
    }],
    options=get_config("code")
)

print("Code Generation (temp=0):")
print(code_response["message"]["content"])

Code Generation (temp=0):
```
def is_palindrome(s):
    return s == s[::-1]


In [41]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  USECASE 2: Creative Brainstorming                           ║
# ║  Goal: Many diverse, surprising ideas                       ║
# ╚══════════════════════════════════════════════════════════════╝

creative_response = client.chat(
    model=MODEL,
    messages=[
        {
        "role": "system",
        "content": "You are a creative director known for unconventional ideas. Think outside the box."
    },{
        "role": "user",
        "content": "Give me 5 unusual marketing ideas for a bookstore"
    }],
    options=get_config("creative")
)

print("USECASE 2 — Creative Brainstorming (temp=1.2, top_p=0.95):")
print(creative_response["message"]["content"])

USECASE 2 — Creative Brainstorming (temp=1.2, top_p=0.95):
What a delightful challenge! Here are 5 unusual marketing ideas to lure book lovers and non-bookworms alike into your beloved bookstore:

1. **Blind Date with a Book**: Wrap several books in brown paper and tie them with string, eliminating any visible cues (title, author, cover art). Customers purchase the "blind date" without knowing what's inside. The thrill of discovery becomes a social media spectacle as customers share their findings on Instagram, Facebook, and Twitter. Share testimonials and behind-the-scenes peeks at your book-wrapping process to build anticipation.

2. **Literary Scavenger Hunt**: Design an immersive, interactive experience by creating a city-wide scavenger hunt inspired by famous books and authors. Hide cryptic clues, quotes, or references around the neighborhood, encouraging participants to decipher their way through famous works (e.g., Harry Potter trail). The final clue leads participants back to y

In [45]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  USECASE 3: Factual Q&A / Customer Support                   ║
# ║  Goal: Accurate, consistent, concise                        ║
# ╚══════════════════════════════════════════════════════════════╝

qa_response = client.chat(
    model=MODEL,
    messages=[
        {
        "role": "system",
        "content": "You are a helpful customer support agent. Give direct, accurate answers. Be concise."
    },{
        "role": "user",
        "content": "What is the difference between RAM and ROM?"
    }],
    options=get_config("qa")
)

print("USECASE 3 — Factual Q&A (temp=0.2, top_p=0.8):")
print(qa_response["message"]["content"])

USECASE 3 — Factual Q&A (temp=0.2, top_p=0.8):
RAM (Random Access Memory) and ROM (Read-Only Memory) are two types of computer memory.


In [58]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  USECASE 4: Data Extraction / Classification                 ║
# ║  Goal: Structured output, machine-readable                  ║
# ╚══════════════════════════════════════════════════════════════╝

review = "The laptop is super fast and the battery lasts all day. However, the keyboard feels cheap and the screen is a bit dim."

extract_response = client.chat(
    model=MODEL,
    messages=[
        {
        "role": "system",
        "content": "Extract sentiment from product reviews. Return ONLY valid JSON with keys: overall_sentiment, positives (list), negatives (list)."
    },{
        "role": "user",
        "content": review
    }],
    options=get_config("extraction")
)

print("USECASE 4 — Data Extraction (temp=0, stop at '}'):")
print(extract_response["message"]["content"])

📋 USECASE 4 — Data Extraction (temp=0, stop at '}'):
Here is the extracted sentiment in JSON format:

```
{
  "overall_sentiment": "NEUTRAL",
  "positives": ["super fast", "battery lasts all day"],
  "negatives": ["keyboard feels cheap", "screen is a bit dim"]
}
```


---
## Parameter Cheat Sheet

| Parameter | Range | Low = | High = | Use for |
|-----------|-------|-------|--------|---------|
| `temperature` | 0.0 – 2.0 | Deterministic | Random/creative | Tune first |
| `top_p` | 0.0 – 1.0 | Focused | Diverse | Fine-tune after temp |
| `top_k` | 1 – ∞ | Greedy | Diverse | Claude/Gemini |
| `max_tokens` | 1 – model limit | Short output | Long output | Always set! |
| `stop_sequences` | list of strings | — | — | Parsing/structure |

## Key Takeaways

1. **Temperature** is your primary creativity dial — start here
2. **Top-P** and **Top-K** refine the token pool — usually only adjust one
3. **Max tokens** controls cost and output length — always set it!
4. **Stop sequences** are essential for structured/parseable output
6. **Context window** determines how much history the model can use
7. For production: use `temperature=0` for deterministic tasks, `~1.0` for creative ones